In [2]:
import diopy
import numpy as np
import pandas as pd
import scanpy as sc 
import anndata as ad
import scanpy.external as sce
import matplotlib.pyplot as plt
import metatime
from metatime import config
from metatime import mecmapper
from metatime import mecs
from metatime import annotator
import pickle

In [2]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

/bigdata/zlin/miniconda3/envs/scrna/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2023-05-20 15:17:36.699074: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-05-20 15:17:36.879817: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-05-20 15:17:37.513582: W tensorflow/compiler/xla/stream_executor/platf

scanpy==1.9.3 anndata==0.9.1 umap==0.5.3 numpy==1.21.6 scipy==1.9.1 pandas==1.5.3 scikit-learn==1.2.2 statsmodels==0.13.5 python-igraph==0.10.4 louvain==0.8.0 pynndescent==0.5.10


In [3]:
becker = diopy.input.read_h5(file='/bigdata/zlin/Melanoma_meta/data/SKCM_Becker/processed_major_after.h5')
franken = diopy.input.read_h5(file='/bigdata/zlin/Melanoma_meta/data/HNSC_Franken/processed_major_after.h5')
bassez1 = diopy.input.read_h5(file='/bigdata/zlin/Melanoma_meta/data/BRCA_Bassez1/filtered.h5')
bassez2 = diopy.input.read_h5(file='/bigdata/zlin/Melanoma_meta/data/BRCA_Bassez2/filtered.h5')
yost = diopy.input.read_h5(file='/bigdata/zlin/Melanoma_meta/data/BCC_Yost/filtered.h5')
crc_li = diopy.input.read_h5(file='/bigdata/zlin/Melanoma_meta/data/CRC_Li/processed_major_after.h5')
tnbc_li = diopy.input.read_h5(file = '/bigdata/zlin/Melanoma_meta/data/TNBC_Li/processed_major_after.h5')

In [4]:
def counts_reverse(adata):
    adata.X = adata.layers['counts'].copy()
    return adata
counts_reverse(bassez1)
counts_reverse(bassez2)
counts_reverse(yost)

AnnData object with n_obs × n_vars = 51518 × 23076
    obs: 'orig.ident', 'patient', 'time_point', 'sort', 'celltype_orig', 'UMAP1', 'UMAP2', 'nCount_RNA', 'nFeature_RNA', 'sample', 'response', 'dataset', 'platform', 'site', 'batch', 'S.Score', 'G2M.Score', 'Phase', 'CC.Difference', 'celltype_bped_main', 'celltype_bped_fine', 'immune', 'celltype_major', 'cell_keep'
    var: 'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable'
    layers: 'counts'

In [5]:
becker = becker[~becker.obs['celltype_major'].isin(['Melanoma','Epithelium'])]
becker.obs['celltype_major'].value_counts()

T_cells        10829
Myeloids        2827
Endothelium     1674
Fibroblasts     1423
B_cells         1210
NK_cells         623
Plasma           392
pDC              194
Name: celltype_major, dtype: int64

In [6]:
franken = franken[~franken.obs['celltype_major'].isin(['Epi/Malignant'])]
franken.obs['celltype_major'].value_counts()

T_cells           78785
Myeloids          73030
Fibroblasts       34998
Plasma            12380
Neutrophils       11698
Endothelium       10481
B_cells            5312
NK_cells           4943
pDC                2681
Mast               1633
Myofibroblasts     1047
Name: celltype_major, dtype: int64

In [15]:
bassez1 = bassez1[~bassez1.obs['celltype_major'].isin(['Malignant'])]
bassez1.obs['celltype_major'].value_counts()

T_cells        48499
Fibroblasts    22418
Myeloids       14983
B_cells        11496
Endothelium     6098
NK_cells        2286
Mast            1068
pDC              696
Name: celltype_major, dtype: int64

In [7]:
bassez2 = bassez2[~bassez2.obs['celltype_major'].isin(['Malignant'])]
bassez2.obs['celltype_major'].value_counts()

T_cells        13222
Fibroblasts    11432
Myeloids        6491
Endothelium     5356
B_cells         1620
NK_cells        1433
pDC              194
Mast              59
Name: celltype_major, dtype: int64

In [8]:
yost = yost[~yost.obs['celltype_major'].isin(['Malignant','Melanocytes'])]
yost.obs['celltype_major'].value_counts()

T_cells           31848
B_cells            6013
Myeloids           3499
Plasma             2624
Fibroblasts        1060
pDC                 979
NK_cells            940
Endothelium         464
Myofibroblasts      365
Name: celltype_major, dtype: int64

In [9]:
crc_li = crc_li[~crc_li.obs['celltype_major'].isin(['Epi/Malignant'])]
crc_li.obs['celltype_major'].value_counts()

T_cells           4260
Endothelium       2576
Plasma            2253
B_cells           1985
Myeloids          1791
Fibroblasts        390
NK_cells           385
Myofibroblasts     311
Mast                37
pDC                 29
Name: celltype_major, dtype: int64

In [19]:
# # Load the pre-trained MeCs
# mecmodel = mecs.MetatimeMecs.load_mec_precomputed()
# # Load functional annotation for MetaTiME-TME
# mectable = mecs.load_mecname(mecDIR = config.SCMECDIR, mode ='table' )
# mecnamedict = mecs.getmecnamedict_ct(mectable) 

In [ ]:
# datasets_raw = [yost, crc_li, tnbc_li, becker, bassez1, bassez2, franken]

In [55]:
# datasets = [yost[:,mecmodel.mec_score.index.intersection(yost.var_names)], 
#             crc_li[:,mecmodel.mec_score.index.intersection(crc_li.var_names)], 
#             tnbc_li[:,mecmodel.mec_score.index.intersection(tnbc_li.var_names)], 
#             becker[:,mecmodel.mec_score.index.intersection(becker.var_names)], 
#             bassez1[:,mecmodel.mec_score.index.intersection(bassez1.var_names)], 
#             bassez2[:,mecmodel.mec_score.index.intersection(bassez2.var_names)], 
#             franken[:,mecmodel.mec_score.index.intersection(franken.var_names)]]

In [ ]:
# pred_list = list()
# for adata in datasets:
#     adata.layers['counts'] = adata.X.copy()
#     sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
#     sc.pp.normalize_total(adata)
#     sc.pp.log1p(adata)
#     adata.raw = adata
#     sc.pp.highly_variable_genes(adata, n_top_genes = 2000, subset=False, layer='counts', flavor='seurat_v3')
#     sc.pp.regress_out(adata, ['total_counts'])
#     sc.pp.scale(adata, max_value=10)
#     sc.pp.pca(adata, svd_solver='arpack')
#     sce.pp.bbknn(adata, batch_key='sample')
#     sc.tl.umap(adata)
#     adata = annotator.overcluster(adata)
#     pdata = mecmapper.projectMecAnn(adata, mecmodel.mec_score)
#     projmat, mecscores = annotator.pdataToTable(pdata, mectable, gcol = 'overcluster')
#     projmat, gpred, gpreddict = annotator.annotator(projmat, mecnamedict, gcol = 'overcluster')
#     pred_list = pred_list.append(pd.concat([projmat['MetaTiME_overcluster'].str.split(': ').str.get(1).str.split('_').str.get(1), mecscores], axis=1))


In [3]:
with open('/bigdata/zlin/Melanoma_meta/data/pred_MetaTiME.pkl', 'rb') as file:
    pred_list = pickle.load(file)